In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from global_parameters import Assumptions

# Creating combinations of planforms

In [ ]:
span_max = 4. #TODO change

lists_to_recombine = dict()

lists_to_recombine['aspect_ratio'] = [5., 10., 17., 27.]
lists_to_recombine['taper'] = [1.]
lists_to_recombine['thickness_to_chord'] = [.06, .12, .18] #NOTE not super justified
lists_to_recombine['LE_offset'] = [-.1, .5] # m #NOTE not super justified
lists_to_recombine['cl_alpha'] = [2*np.pi]
lists_to_recombine['cl_max'] = [1.]
lists_to_recombine['cm_ac'] = [-.05, 0.05] #TODO change
lists_to_recombine['cl_0'] = [0.]
lists_to_recombine['pf_type'] = ['tail', 'canard']

In [3]:
ltr_keys = lists_to_recombine.keys()
ltr_values = lists_to_recombine.values()

planforms_raw = list(itt.product(*ltr_values))
print(planforms_raw)

planform_params = list()
for planform_raw in planforms_raw:
    planform_param = dict()
    for i, key in enumerate(ltr_keys):
        planform_param[key] = planform_raw[i]
    planform_params.append(planform_param)

assert len(planform_params) == 96, len(planform_params)

[(5.0, 1.0, 0.06, -0.2, 6.283185307179586, 1.0, -0.05, 0.0, 'tail'), (5.0, 1.0, 0.06, -0.2, 6.283185307179586, 1.0, -0.05, 0.0, 'canard'), (5.0, 1.0, 0.06, -0.2, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.06, -0.2, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.06, 0.4, 6.283185307179586, 1.0, -0.05, 0.0, 'tail'), (5.0, 1.0, 0.06, 0.4, 6.283185307179586, 1.0, -0.05, 0.0, 'canard'), (5.0, 1.0, 0.06, 0.4, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.06, 0.4, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, -0.2, 6.283185307179586, 1.0, -0.05, 0.0, 'tail'), (5.0, 1.0, 0.12, -0.2, 6.283185307179586, 1.0, -0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, -0.2, 6.283185307179586, 1.0, 0.05, 0.0, 'tail'), (5.0, 1.0, 0.12, -0.2, 6.283185307179586, 1.0, 0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, 0.4, 6.283185307179586, 1.0, -0.05, 0.0, 'tail'), (5.0, 1.0, 0.12, 0.4, 6.283185307179586, 1.0, -0.05, 0.0, 'canard'), (5.0, 1.0, 0.12, 0.4, 6.283185307179586, 1.0,

In [4]:
wing_area = span_max**2/max(lists_to_recombine['aspect_ratio'])

planforms:list[tuple[Planform, str, bool]] = list()

for planform_param in planform_params:
    span = np.sqrt(wing_area * planform_param['aspect_ratio'])

    planforms.append((Planform(
        aspect_ratio=planform_param['aspect_ratio'],
        taper=planform_param['taper'],
        sweep_quarter_deg=np.rad2deg(np.arctan(planform_param["LE_offset"] / span * 2)),
        thickness_to_chord=planform_param['thickness_to_chord'],
        cm_quarter_chord=planform_param['cm_ac'],
        cl0=planform_param['cl_0'],
        clmax=planform_param['cl_max'],
        flap=False, #NOTE for now
        airfoil_lift_slope=planform_param['cl_alpha'],
        wetted_surface_ratio=1.07,
        interference_factor=1.,
        span=span
    ), planform_param["pf_type"]))

In [5]:
print(planforms)

[(<Aircraft.Planform.Planform object at 0x000001F7835A2150>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7835A1E50>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B65E80>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B669F0>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B665A0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B66B10>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B65B50>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B65C70>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B662D0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B66570>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B66690>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B666C0>, 'canard'), (<Aircraft.Planform.Planform object at 0x000001F7D4B666F0>, 'tail'), (<Aircraft.Planform.Planform object at 0x000001F7D4B66720>, 'canard'), (<Aircraft.Planform

# Caching Planform properties

## CD0

In [6]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

for planform in planforms:
    planform[0].add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    planform[0].add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
    #NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    planform[0].add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    planform[0].add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

## Planform Mass and Cg

In [7]:
#TODO change to an actual structural weight estimation
for planform in planforms:
    planform[0].mass_cache = 5.
    planform[0].x_cg_cache = planform[0].x_MAC + planform[0].MAC/3

# Saving the planforms

In [8]:
with open("pickles/planform_pickle_official.pcl", "w+b") as f:
    pickle.dump(planforms, f)

### Recovery to see if pickled correctly

In [9]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered = pickle.load(f)

In [10]:
sample:Planform = plaforms_recovered[5][0]
print(f"AR: {sample.aspect_ratio}")
print(f"cr: {sample.c_root}")
print(f"CD0 takeoff: {sample.CD0_cache["takeoff"]}")

AR: 5.0
cr: 0.34426518632954817
CD0 takeoff: 0.0038482869239055283
